# DPD Experiment Analysis

Analysis of autonomous DPD research results from `results.tsv`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load the TSV (tab-separated, 5 columns: commit, nmse_db, acpr_after_dbc, status, description)
df = pd.read_csv("results.tsv", sep="\t")
df["nmse_db"] = pd.to_numeric(df["nmse_db"], errors="coerce")
df["acpr_after_dbc"] = pd.to_numeric(df["acpr_after_dbc"], errors="coerce")
df["status"] = df["status"].str.strip().str.upper()

print(f"Total experiments: {len(df)}")
print(f"Columns: {list(df.columns)}")
df.head(10)

In [ ]:
counts = df["status"].value_counts()
print("Experiment outcomes:")
print(counts.to_string())

n_keep = counts.get("KEEP", 0)
n_discard = counts.get("DISCARD", 0)
n_crash = counts.get("CRASH", 0)
n_decided = n_keep + n_discard
if n_decided > 0:
    print(f"\nKeep rate: {n_keep}/{n_decided} = {n_keep / n_decided:.1%}")

In [ ]:
# Show all KEPT experiments (the improvements that stuck)
kept = df[df["status"] == "KEEP"].copy()
print(f"KEPT experiments ({len(kept)} total):\n")
for i, row in kept.iterrows():
    nmse = row["nmse_db"]
    acpr = row["acpr_after_dbc"]
    desc = row["description"]
    print(f"  #{i:3d}  NMSE={nmse:.2f} dB  ACPR={acpr:.2f} dBc  {desc}")

## NMSE Over Time

Track how the best (kept) NMSE evolves as experiments progress. The running minimum shows the frontier — the best linearization achieved so far. More negative = better.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))

# Filter out crashes for plotting
valid = df[df["status"] != "CRASH"].copy()
valid = valid.reset_index(drop=True)

baseline_nmse = valid.loc[0, "nmse_db"]

# Only plot points near or below baseline (the interesting region)
below = valid[valid["nmse_db"] <= baseline_nmse + 1.0]

# Plot discarded as faint background dots
disc = below[below["status"] == "DISCARD"]
ax.scatter(disc.index, disc["nmse_db"],
           c="#cccccc", s=12, alpha=0.5, zorder=2, label="Discarded")

# Plot kept experiments as prominent green dots
kept_v = below[below["status"] == "KEEP"]
ax.scatter(kept_v.index, kept_v["nmse_db"],
           c="#2ecc71", s=50, zorder=4, label="Kept", edgecolors="black", linewidths=0.5)

# Running minimum step line
kept_mask = valid["status"] == "KEEP"
kept_idx = valid.index[kept_mask]
kept_nmse = valid.loc[kept_mask, "nmse_db"]
running_min = kept_nmse.cummin()
ax.step(kept_idx, running_min, where="post", color="#27ae60",
        linewidth=2, alpha=0.7, zorder=3, label="Running best")

# Label each kept experiment with its description
for idx, nmse in zip(kept_idx, kept_nmse):
    desc = str(valid.loc[idx, "description"]).strip()
    if len(desc) > 45:
        desc = desc[:42] + "..."

    ax.annotate(desc, (idx, nmse),
                textcoords="offset points",
                xytext=(6, -10), fontsize=8.0,
                color="#1a7a3a", alpha=0.9,
                rotation=-20, ha="left", va="top")

n_total = len(df)
n_kept = len(df[df["status"] == "KEEP"])
ax.set_xlabel("Experiment #", fontsize=12)
ax.set_ylabel("NMSE (dB, more negative = better)", fontsize=12)
ax.set_title(f"DPD Research Progress: {n_total} Experiments, {n_kept} Kept Improvements", fontsize=14)
ax.legend(loc="upper right", fontsize=9)
ax.grid(True, alpha=0.2)

# Y-axis: from best (most negative) to baseline
best_nmse = kept_nmse.min()
margin = abs(baseline_nmse - best_nmse) * 0.15
ax.set_ylim(best_nmse - margin, baseline_nmse + margin)

plt.tight_layout()
plt.savefig("progress.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to progress.png")

## Summary Statistics

In [ ]:
# Summary stats
kept = df[df["status"] == "KEEP"].copy()
baseline_nmse = df.iloc[0]["nmse_db"]
best_nmse = kept["nmse_db"].min()
best_row = kept.loc[kept["nmse_db"].idxmin()]

print(f"Baseline NMSE:     {baseline_nmse:.2f} dB")
print(f"Best NMSE:         {best_nmse:.2f} dB")
print(f"Total improvement: {baseline_nmse - best_nmse:.2f} dB")
print(f"Best experiment:   {best_row['description']}")
print()

# How many experiments to find each improvement
print("Cumulative effort per improvement:")
kept_sorted = kept.reset_index()
for i, (_, row) in enumerate(kept_sorted.iterrows()):
    desc = str(row["description"]).strip()
    print(f"  Experiment #{row['index']:3d}: NMSE={row['nmse_db']:.2f} dB  ACPR={row['acpr_after_dbc']:.2f} dBc  {desc}")

## Top Hits (Kept Experiments by Improvement)

In [ ]:
# Each kept experiment's delta vs the previous kept experiment
kept = df[df["status"] == "KEEP"].copy()
kept["prev_nmse"] = kept["nmse_db"].shift(1)
kept["delta"] = kept["prev_nmse"] - kept["nmse_db"]  # positive = improvement

# Drop baseline (no delta)
hits = kept.iloc[1:].copy()

# Sort by delta improvement (biggest first)
hits = hits.sort_values("delta", ascending=False)

print(f"{'Rank':>4}  {'Delta':>8}  {'NMSE':>10}  Description")
print("-" * 80)
for rank, (_, row) in enumerate(hits.iterrows(), 1):
    print(f"{rank:4d}  {row['delta']:+.2f} dB  {row['nmse_db']:.2f} dB  {row['description']}")

print(f"\n{'':>4}  {hits['delta'].sum():+.2f} dB  {'':>10}  TOTAL improvement over baseline")

## ACPR Comparison

In [ ]:
# Plot NMSE vs ACPR for kept experiments
kept = df[df["status"] == "KEEP"].copy()

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(kept["nmse_db"], kept["acpr_after_dbc"],
           c="#2ecc71", s=60, edgecolors="black", linewidths=0.5, zorder=3)

for _, row in kept.iterrows():
    desc = str(row["description"]).strip()
    if len(desc) > 30:
        desc = desc[:27] + "..."
    ax.annotate(desc, (row["nmse_db"], row["acpr_after_dbc"]),
                textcoords="offset points", xytext=(6, 6),
                fontsize=7, alpha=0.8)

ax.set_xlabel("NMSE (dB)", fontsize=12)
ax.set_ylabel("ACPR (dBc)", fontsize=12)
ax.set_title("NMSE vs ACPR (both: more negative = better)", fontsize=14)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()